# risk-management_005

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (236).ipynb`

| Field | Value |
|---|---|
| Section | `risk_management` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `daily` |
| Code cells | 6 |
| Detected functions | compute_drawdown |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
import pandas as pd
import plotly.express as px

# === Load tradebooks ===
df1 = pd.read_csv("/content/drive/MyDrive/Copy of tradebook_FOREX_JMA.csv", parse_dates=["Entry Time"])
df2 = pd.read_csv("/content/drive/MyDrive/Copy of tradebook_FOREX_LTPKAMADSL.csv", parse_dates=["Entry Time"])

# === Standardize tickers ===
df1["Ticker"] = df1["Ticker"].str.upper()
df2["Ticker"] = df2["Ticker"].str.upper()

# === Extract date only ===
df1["Date"] = df1["Entry Time"].dt.date
df2["Date"] = df2["Entry Time"].dt.date

# === Group by Date and sum PnL ===
pnl1_daily = df1.groupby("Date")["PnL"].sum().rename("PnL_1")
pnl2_daily = df2.groupby("Date")["PnL"].sum().rename("PnL_2")

# === Combine into single DataFrame ===
daily_pnl = pd.concat([pnl1_daily, pnl2_daily], axis=1).dropna()

# === Calculate rolling correlation (optional) ===
# rolling_corr = daily_pnl['PnL_1'].rolling(30).corr(daily_pnl['PnL_2'])

# === Correlation matrix (across columns) ===
correlation_matrix = daily_pnl.corr()

# === Plot static 2x2 heatmap of daily PnL correlation ===
fig = px.imshow(
    correlation_matrix,
    text_auto=True,
    color_continuous_scale='RdBu',
    title="Daily PnL Correlation Between Strategies",
    labels=dict(x="Strategy", y="Strategy", color="Correlation")
)

fig.update_layout(width=500, height=500)
fig.show()


In [ ]:
import plotly.graph_objects as go

# 30-day rolling correlation
daily_pnl["Rolling Corr"] = daily_pnl["PnL_1"].rolling(30).corr(daily_pnl["PnL_2"])

# Plot
fig = go.Figure()
fig.add_trace(go.Scatter(x=daily_pnl.index, y=daily_pnl["Rolling Corr"],
                         mode='lines', name='30D Rolling Corr'))
fig.update_layout(title="Rolling 30-Day Correlation Between Strategies",
                  xaxis_title="Date", yaxis_title="Correlation",
                  yaxis=dict(range=[-1, 1]))
fig.show()


In [ ]:
# Group each tradebook by Ticker and compute total PnL
pnl1_by_ticker = df1.groupby("Ticker")["PnL"].sum().rename("PnL_1")
pnl2_by_ticker = df2.groupby("Ticker")["PnL"].sum().rename("PnL_2")

# Join the two
ticker_df = pd.concat([pnl1_by_ticker, pnl2_by_ticker], axis=1).dropna()

# Compute correlation
ticker_corr = ticker_df.corr().iloc[0, 1]
print("Overall correlation by Ticker PnL:", ticker_corr)

# Plot using Plotly
import plotly.express as px
fig = px.imshow(ticker_df.corr(), text_auto=True, color_continuous_scale='RdBu',
                title="Correlation Between Strategy PnLs by Ticker")
fig.show()


In [ ]:
# Compute daily volatility (rolling std)
daily_pnl["Vol_1"] = daily_pnl["PnL_1"].rolling(10).std()
daily_pnl["Vol_2"] = daily_pnl["PnL_2"].rolling(10).std()
daily_pnl["Rolling Corr"] = daily_pnl["PnL_1"].rolling(30).corr(daily_pnl["PnL_2"])

# Scatterplot: Correlation vs Volatility
import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=daily_pnl["Vol_1"],
    y=daily_pnl["Rolling Corr"],
    mode="markers",
    name="Volatility vs Correlation"
))
fig.update_layout(title="Volatility vs Rolling Correlation (Strategy 1)",
                  xaxis_title="10D Rolling Volatility (Strategy 1)",
                  yaxis_title="30D Rolling Correlation")
fig.show()


In [ ]:
# Compute cumulative PnL
daily_pnl["CumPnL_1"] = daily_pnl["PnL_1"].cumsum()
daily_pnl["CumPnL_2"] = daily_pnl["PnL_2"].cumsum()

# Compute drawdown
def compute_drawdown(series):
    peak = series.cummax()
    return (series - peak) / peak

daily_pnl["DD_1"] = compute_drawdown(daily_pnl["CumPnL_1"])
daily_pnl["DD_2"] = compute_drawdown(daily_pnl["CumPnL_2"])

# Scatterplot: Drawdown vs Rolling Correlation
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=daily_pnl["DD_1"],
    y=daily_pnl["Rolling Corr"],
    mode="markers",
    name="Drawdown vs Correlation"
))
fig.update_layout(title="Drawdown vs Rolling Correlation (Strategy 1)",
                  xaxis_title="Drawdown (Strategy 1)",
                  yaxis_title="30D Rolling Correlation")
fig.show()


In [ ]:
summary = {
    "Daily Corr": round(daily_pnl["PnL_1"].corr(daily_pnl["PnL_2"]), 4),
    "Ticker Corr": round(ticker_corr, 4),
    "Avg Rolling Corr": round(daily_pnl["Rolling Corr"].mean(), 4),
    "Min Rolling Corr": round(daily_pnl["Rolling Corr"].min(), 4),
    "Max Rolling Corr": round(daily_pnl["Rolling Corr"].max(), 4),
    "Mean Drawdown 1": round(daily_pnl["DD_1"].mean(), 4),
    "Mean Drawdown 2": round(daily_pnl["DD_2"].mean(), 4),
    "Strategy 1 Volatility": round(daily_pnl["PnL_1"].std(), 4),
    "Strategy 2 Volatility": round(daily_pnl["PnL_2"].std(), 4)
}

summary_df = pd.DataFrame.from_dict(summary, orient='index', columns=["Value"])
print(summary_df)
